<a href="https://colab.research.google.com/github/LAB-FAM/nice-rag-project/blob/main/colab/query_understanding_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. INSTALL REQUIRED LIBRARIES
!pip install -q langgraph langchain langchain-community langchain-ollama chromadb pydantic sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6

In [2]:
# 2. SETUP ENVIRONMENT (Vector DB Extraction)
import os
import shutil
import zipfile

# 1. Delete previous folders to ensure a clean run
for folder in ["methodology_db", "vector_db"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

# 2. Extract vector_db.zip to vector_db folder (Phase 0 output)
if os.path.exists("vector_db.zip"):
    print("[*] Extracting vector_db.zip...")
    with zipfile.ZipFile("vector_db.zip", 'r') as zip_ref:
        zip_ref.extractall("vector_db")
    print("[*] Extraction complete.")
else:
    print("[!] Warning: vector_db.zip not found. Make sure you uploaded it.")

[*] Extracting vector_db.zip...
[*] Extraction complete.


In [3]:
# 3. INSTALL AND RUN OLLAMA IN BACKGROUND
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("[*] Starting Ollama server...")
subprocess.Popen(["nohup", "ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

# We will use llama3 as our local reasoning agent
print("[*] Pulling llama3 model (This will take a few minutes)...")
!ollama pull llama3
print("[*] Pulling nomic-embed-text model...")
!ollama pull nomic-embed-text
print("[*] Setup complete!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
[*] Starting Ollama server...
[*] Pulling llama3 model (This will take a few mi

In [4]:
# 4. LANGGRAPH NODE 1 IMPLEMENTATION
import json
from typing import List, TypedDict, Optional
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

In [5]:
# 4.1. UNIFIED STATE SCHEMA
class UnifiedGraphState(TypedDict):
    research_question: str
    primary_condition: str
    related_conditions: List[str]
    observations: List[str]
    explicit_exclusions: List[str]
    search_terms: List[str]
    concept_type: str
    snomed_top_hierarchy: str
    suggested_validation_sources: List[str]
    ambiguity_notes: str
    qof_rules_text: str
    relevant_guidelines: List[str]
    candidate_codes: List[dict]
    final_codes: List[dict]

In [12]:
# 4.2. PYDANTIC: LLM STRUCTURED OUTPUT GATEKEEPER
# Forces the LLM to return exactly this JSON structure
class QueryExtraction(BaseModel):
    primary_condition: str = Field(description="The main disease or clinical condition (e.g., 'Type 2 Diabetes')")
    related_conditions: List[str] = Field(description="Other mentioned conditions.", default=[])
    observations: List[str] = Field(description="CRITICAL: Any laboratory tests, measurements (like BMI, 'HbA1c', 'Blood Pressure'), or metrics.", default=[])
    explicit_exclusions: List[str] = Field(description="Conditions or keywords that must be explicitly excluded.", default=[])
    search_terms: List[str] = Field(description="CRITICAL: Provide exactly 2 or 3 official UK NHS/SNOMED synonyms for the primary condition. DO NOT include clinical modifiers like 'newly diagnosed', 'history of', or 'suspected'.", default=[])
    qof_domain_prefix: str = Field(description="The standard UK QOF domain abbreviation for the condition (e.g., 'DM' for Diabetes, 'HYP' for Hypertension, 'OB' for Obesity, 'AST' for Asthma). If unknown, leave empty.", default="")
    concept_type: str = Field(description="SNOMED concept type (e.g., 'disease', 'finding', 'procedure').", default="")
    snomed_top_hierarchy: str = Field(description="Top level SNOMED hierarchy category (e.g., 'Clinical finding').", default="")
    relevant_guidelines: List[str] = Field(description="Short names of inferred guidelines (e.g., ['NG28', 'QOF 2025/26'])", default_factory=list)
    suggested_validation_sources: List[str] = Field(description="Suggested sources for validation (e.g., ['OpenCodelists']).", default=[])
    ambiguity_notes: str = Field(description="Any missing or ambiguous info in the user query.", default="")

In [14]:
# 4.3. INITIALIZE MODELS & VECTOR DATABASE

# Connect to the offline vector database created in Phase 0
print("[*] Initializing local models & connecting to ChromaDB...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_db = Chroma(persist_directory="vector_db/methodology_db", embedding_function=embeddings)

# Initialize the local LLM with structured output mapping
llm = ChatOllama(model="llama3", temperature=0)
structured_llm = llm.with_structured_output(QueryExtraction)

[*] Initializing local models & connecting to ChromaDB...


In [15]:
# 4.4. NODE 1 MAIN FUNCTION: QUERY UNDERSTANDING & RAG

def node_1_query_understanding(state: UnifiedGraphState) -> UnifiedGraphState:
    """
    Node 1 parses the user's research question into structured clinical entities,
    and retrieves the corresponding official QOF/NICE rules from the local Vector DB.
    """
    print("\n--- NODE 1 - QUERY UNDERSTANDING & METHODOLOGY RAG STARTED ---")

    query = state["research_question"]
    print(f"[*] Processing Query: '{query}'")

    # STEP A: LLM DECOMPOSITION
    print("[*] Decomposing query via structured LLM...")
    system_prompt = """You are an expert clinical data analyst extracting key entities from a user query for SNOMED CT mapping.
    CRITICAL INSTRUCTION: You MUST extract all relevant information and fill out every field. Do not leave fields empty if the information is present or can be safely inferred from clinical context.

    EXAMPLE EXACT BEHAVIOR:
    User: "Find active SNOMED codes for adult patients with Type 2 Diabetes who also suffer from Essential Hypertension and have a BMI of 30 or above. Exclude suspected cases and gestational diabetes."
    Output:
    {{
        "primary_condition": "Type 2 Diabetes Mellitus",
        "related_conditions": ["Essential Hypertension"],
        "observations": ["Body mass index (BMI)", "Blood Pressure"],
        "explicit_exclusions": ["Suspected case of diabetes", "Gestational diabetes"],
        "search_terms": ["Type 2 diabetes", "Diabetes mellitus type II", "T2DM"],
        "concept_type": "disease",
        "snomed_top_hierarchy": "Clinical finding",
        "suggested_validation_sources": ["OpenCodelists", "QOF Rules"],
        "ambiguity_notes": "Age range 'adult' is broad, standard adult rules apply."
    }}
    """

    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", "Research Question: {question}")
    ])

    # Pipe the prompt to the structured LLM
    chain = prompt_template | structured_llm
    extracted_data = chain.invoke({"question": query})

    print(f"[*] Primary Condition: {extracted_data.primary_condition}")
    print(f"[*] Synonyms/Search Terms: {extracted_data.search_terms}")

    # STEP B: METHODOLOGY RAG (Vector Search - Retrieve Rules from ChromaDB)
    print("[*] Querying local Vector DB for clinical guidelines and QOF rules...")
    qof_rules_payload = ""
    guideline_sources = set() # Store metadata sources dynamically

    # We search the vector DB for both the primary and related conditions
    search_queries = [extracted_data.primary_condition] + extracted_data.related_conditions

    for sq in search_queries:
        print(f"    -> Retrieving official rules for: {sq}")
        if not sq: continue

        # NOTE: Improved search using MMR with QOF Domain Prefix anchoring.
        # Removed "guidelines" to prevent matching bibliography citations in other PDFs.
        # Increased lambda_mult to 0.85 to prioritize relevance over extreme diversity.
        domain_anchor = extracted_data.qof_domain_prefix if extracted_data.qof_domain_prefix else ""
        search_str = f"{sq} {domain_anchor} QOF indicators clinical criteria".strip()

        docs = vector_db.max_marginal_relevance_search(
            query=search_str,
            k=3,
            fetch_k=15,
            lambda_mult=0.85
        )

        for doc in docs:
            # Capture metadata (Source PDF and Page Number) for later justification
            source = doc.metadata.get("source", "Unknown_Source")
            page = doc.metadata.get("page", "N/A")

            # Extract just the filename (e.g., 'NG28_Type2_Diabetes')
            filename = source.split("/")[-1].replace(".pdf", "")
            guideline_sources.add(filename)

            qof_rules_payload += f"--- Source: {filename} (Page {page}) ---\n"
            qof_rules_payload += doc.page_content + "\n\n"

    # STEP C: UPDATE STATE
    state.update({
        "primary_condition": extracted_data.primary_condition,
        "related_conditions": extracted_data.related_conditions,
        "observations": extracted_data.observations,
        "explicit_exclusions": extracted_data.explicit_exclusions,
        "search_terms": extracted_data.search_terms,
        "concept_type": extracted_data.concept_type,
        "snomed_top_hierarchy": extracted_data.snomed_top_hierarchy,
        "suggested_validation_sources": extracted_data.suggested_validation_sources,
        "ambiguity_notes": extracted_data.ambiguity_notes,
        "qof_rules_text": qof_rules_payload,
        "relevant_guidelines": list(guideline_sources)
    })

    print("--- NODE 1 COMPLETE ---")
    return state

In [16]:
# 5. Test Node 1
if __name__ == "__main__":
    # Create an initial mock state to simulate an incoming analyst request
    initial_state: UnifiedGraphState = {
        "research_question": "Looking for SNOMED codes for Essential Hypertension where the latest blood pressure reading is above 140/90. Explicitly exclude secondary hypertension.",
        "primary_condition": "",
        "related_conditions": [],
        "observations": [],
        "concept_type": "",
        "snomed_top_hierarchy": "",
        "search_terms": [],
        "explicit_exclusions": [],
        "relevant_guidelines": [],
        "suggested_validation_sources": [],
        "ambiguity_notes": "",
        "qof_rules_text": "",
        "candidate_codes": [],
        "final_codes": []
    }

    # Execute Node 1
    result_state = node_1_query_understanding(initial_state)

    # Print a summary to verify the state update was successful
    print("\n=== FINAL STATE SUMMARY ===")
    print(f"Primary Condition: {result_state['primary_condition']}")
    print(f"Related Conditions: {result_state['related_conditions']}")
    print(f"Observations: {result_state['observations']}")
    print(f"Exclusions: {result_state['explicit_exclusions']}")
    print(f"Guidelines Cited (Metadata): {result_state['relevant_guidelines']}")
    print(f"\nExtracted QOF Rules Payload:\n{result_state['qof_rules_text']}...")


--- NODE 1 - QUERY UNDERSTANDING & METHODOLOGY RAG STARTED ---
[*] Processing Query: 'Looking for SNOMED codes for Essential Hypertension where the latest blood pressure reading is above 140/90. Explicitly exclude secondary hypertension.'
[*] Decomposing query via structured LLM...
[*] Primary Condition: Essential Hypertension
[*] Synonyms/Search Terms: ['Essential Hypertension', 'Primary Hypertension', 'High Blood Pressure']
[*] Querying local Vector DB for clinical guidelines and QOF rules...
    -> Retrieving official rules for: Essential Hypertension
--- NODE 1 COMPLETE ---

=== FINAL STATE SUMMARY ===
Primary Condition: Essential Hypertension
Related Conditions: []
Observations: ['Blood Pressure']
Exclusions: ['Secondary Hypertension']
Guidelines Cited (Metadata): ['qof_combined']

Extracted QOF Rules Payload:
--- Source: qof_combined (Page 13) ---
DM020 Reporting and verification 
i. See indicator wording for requirement criteria. 
 
DM021 (based on NICE IND180) 
DM021 Rationale

In [ ]:
# 5. COMPREHENSIVE TEST SCENARIOS (TEST SUITE)
research_questions_test_suite = [
    # CATEGORY 1: TYPE 2 DIABETES (T2DM)
    "Identify QOF eligible SNOMED codes for patients newly diagnosed with Type 2 Diabetes Mellitus. Do not include patients with gestational diabetes or diabetes due to cystic fibrosis.",
    #"Patient has a history of Type 2 Diabetes and is currently suffering from Obesity. Find the relevant active SNOMED codes. Exclude suspected cases.",

    # CATEGORY 2: HYPERTENSION
    # "Looking for SNOMED codes for Essential Hypertension where the latest blood pressure reading is above 140/90. Explicitly exclude secondary hypertension.",
    # "Find active clinical codes for hypertensive patients. We must strictly exclude white coat syndrome and pregnancy-induced hypertension.",

    # CATEGORY 3: OBESITY
    # "Find active clinical codes for morbid obesity in adult patients with a Body Mass Index strictly greater than 40.",
    # "Clinical terms for obese patients. Explicitly exclude maternal obesity or obesity complicating pregnancy."
]

# Run the tests
if __name__ == "__main__":
    for i, query in enumerate(research_questions_test_suite, 1):
        print(f"\n{'='*80}")
        print(f"🚀 RUNNING TEST {i}:\nQUERY: {query}")
        print(f"{'='*80}")

        initial_state = {"research_question": query}
        try:
            result_state = node_1_query_understanding(initial_state)

            print(f"\n[LLM EXTRACTIONS]")
            print(f"📍 Primary Condition : {result_state.get('primary_condition')}")
            print(f"🔗 Related Conditions: {result_state.get('related_conditions')}")
            print(f"📊 Observations      : {result_state.get('observations')}")
            print(f"🚫 Exclusions        : {result_state.get('explicit_exclusions')}")
            print(f"🔎 Search Synonyms   : {result_state.get('search_terms')}")

            print(f"📌 Concept Type      : {result_state.get('concept_type')}")
            print(f"🏛️ Hierarchy         : {result_state.get('snomed_top_hierarchy')}")
            print(f"💡 Validation Sources: {result_state.get('suggested_validation_sources')}")
            print(f"⚠️ Ambiguity Notes   : {result_state.get('ambiguity_notes')}")

            print(f"📚 Cited Guidelines  : {result_state.get('relevant_guidelines')}")

            print(f"\n[EXTRACTED QOF RULES PAYLOAD (FULL TEXT)]")
            print(result_state.get('qof_rules_text'))

        except Exception as e:
            print(f"❌ TEST FAILED: {e}")


🚀 RUNNING TEST 1:
QUERY: Identify QOF eligible SNOMED codes for patients newly diagnosed with Type 2 Diabetes Mellitus. Do not include patients with gestational diabetes or diabetes due to cystic fibrosis.

--- NODE 1 - QUERY UNDERSTANDING & METHODOLOGY RAG STARTED ---
[*] Processing Query: 'Identify QOF eligible SNOMED codes for patients newly diagnosed with Type 2 Diabetes Mellitus. Do not include patients with gestational diabetes or diabetes due to cystic fibrosis.'
[*] Decomposing query via structured LLM...
[*] Primary Condition: Type 2 Diabetes Mellitus
[*] Synonyms/Search Terms: ['Type 2 diabetes', 'Diabetes mellitus type II', 'T2DM', 'Newly diagnosed']
[*] Querying local Vector DB for clinical guidelines and QOF rules...
    -> Retrieving official rules for: Type 2 Diabetes Mellitus
    -> Retrieving official rules for: Fasting plasma glucose
    -> Retrieving official rules for: HbA1c
--- NODE 1 COMPLETE ---

[LLM EXTRACTIONS]
📍 Primary Condition : Type 2 Diabetes Mellitus
🔗